# S09 toy — the honest debrief, checked

Companion lesson: [S09 — Evidence reports](../lessons/S09-evidence-reports.html).

A DIY-assistant session transcript, a report generator, and the two validators that keep the report honest. No network, no keys, no cost. The "report writer" here is a plain Python function — the failure modes (fabricated citation, dropped safety event, rounded-up outcome) are exactly the ones real LLM summarizers commit, so the validators you build here are the ones production needs.

**How to use:** run cells in order. For each experiment, write your prediction as a comment *before* running. The gap between prediction and result is the lesson.

## The raw material: transcript + run record

Two artifacts come out of every run, and both are ground truth:

- **the message list** — what was actually said, with per-call usage attached (API-shaped, like S01);
- **the run record** — what the harness logged *during* the run: typed events with **stable ids** and verbatim excerpts, plus the stop reason. (This is the S08 tracing layer's output.)

The report generator reads both. The validators read both. **Read the transcript once, now** — later you will test whether the report can stand in for the reread.

In [ ]:
# A DIY-assistant session: mounting a bookshelf on a drywall wall.
# (The scenario details — here and in the experiment-5 mirror fixture — are
#  illustrative teaching material, not mounting advice.)
MESSAGES = [
    {"turn": 1, "role": "user",
     "content": "I need to get this bookshelf up above my desk before Friday. "
                "Drywall, no studs where I want it. Where do I start?",
     "usage": {"total_tokens": 31, "latency_ms": 0}},
    {"turn": 2, "role": "assistant",
     "content": "Start with anchors, not screws. For drywall without studs, snap "
                "toggles are the right default. How heavy is the shelf, loaded?",
     "usage": {"total_tokens": 214, "latency_ms": 940}},
    {"turn": 3, "role": "user",
     "content": "Maybe 8 kg empty, another 15 in books. What anchors exactly?",
     "usage": {"total_tokens": 18, "latency_ms": 0}},
    {"turn": 4, "role": "assistant",
     "content": "Four snap toggles rated 25 kg each gives you 4x headroom. Mark a "
                "level line with painter's tape, then drill the two end positions first.",
     "usage": {"total_tokens": 238, "latency_ms": 1120}},
    {"turn": 5, "role": "user",
     "content": "Marked and drilling. The bit is smoking a little - is that normal?",
     "usage": {"total_tokens": 16, "latency_ms": 0}},
    {"turn": 6, "role": "assistant",
     "content": "Stop drilling. Smoke means metal, not wood. Press a strong magnet "
                "to the spot: if it sticks, that is a steel stud or plate, not an anchor point.",
     "usage": {"total_tokens": 251, "latency_ms": 1030}},
    {"turn": 7, "role": "user",
     "content": "The magnet sticks, so it is a stud. I will just drill into it - "
                "a wire is probably not in there, right?",
     "usage": {"total_tokens": 27, "latency_ms": 0}},
    {"turn": 8, "role": "assistant",
     "content": "Do not guess at that. Cables often run vertically from outlets, and "
                "'probably' is how people find them. Confirm the stud edges with a stud "
                "finder before any drilling, and stay clear of outlet height.",
     "usage": {"total_tokens": 289, "latency_ms": 1310}},
    {"turn": 9, "role": "user",
     "content": "Stud finder confirms wood edges, no cable warning. Pilot holes in, "
                "toggles set in the two drywall spots.",
     "usage": {"total_tokens": 24, "latency_ms": 0}},
    {"turn": 10, "role": "assistant",
     "content": "Good. Seat the brackets now: snug the toggle bolts until they bite, "
                "then stop - overtightening crushes drywall.",
     "usage": {"total_tokens": 243, "latency_ms": 980}},
    {"turn": 11, "role": "user",
     "content": "First bracket is on. Second one seated by feel - I did not need "
                "the torque note this time.",
     "usage": {"total_tokens": 21, "latency_ms": 0}},
    {"turn": 12, "role": "assistant",
     "content": "Now the check that matters: level across both brackets, in both "
                "directions, before the shelf goes on.",
     "usage": {"total_tokens": 197, "latency_ms": 870}},
    {"turn": 13, "role": "user",
     "content": "Level reads true both ways. Shelf is up. Loading the books.",
     "usage": {"total_tokens": 15, "latency_ms": 0}},
    {"turn": 14, "role": "assistant",
     "content": "Nicely done. One caution: give the toggles 24 hours to settle before "
                "the heavy books go on, then re-check the level.",
     "usage": {"total_tokens": 226, "latency_ms": 1010}},
]

EVENTS = [
    {"id": "smoking-bit", "turn": 5, "type": "observation",
     "quote": "The bit is smoking a little",
     "note": "user hit something metallic mid-drill"},
    {"id": "wire-guess", "turn": 7, "type": "safety",
     "quote": "a wire is probably not in there",
     "note": "user proposed drilling into an unidentified obstruction on a guess"},
    {"id": "guess-blocked", "turn": 8, "type": "safety_response",
     "quote": "Do not guess at that",
     "note": "assistant blocked the guess; required positive confirmation before drilling"},
    {"id": "torque-internalized", "turn": 11, "type": "milestone",
     "quote": "I did not need the torque note this time",
     "note": "procedure internalized on the second bracket"},
    {"id": "level-true", "turn": 13, "type": "milestone",
     "quote": "Level reads true both ways",
     "note": "acceptance check passed; user loaded the shelf"},
]

RUN = {"session_id": "shelf-001", "stop_reason": "task_complete",
       "turns_used": 14, "max_turns": 20, "model": "mock-diy-v1"}

# The run record vouches for its excerpts: every logged quote must appear
# verbatim in the turn it cites. (If this assert ever fails, the *fixture*
# is lying — fix the fixture before trusting anything downstream.) Each event
# also carries a stable id: coverage (experiment 4) compares event IDENTITIES —
# turn numbers stay for display and citation only.
_by_turn = {m["turn"]: m for m in MESSAGES}
for _e in EVENTS:
    assert _e["quote"] in _by_turn[_e["turn"]]["content"], f"fixture broken at turn {_e['turn']}"
print(f"loaded {len(MESSAGES)} turns, {len(EVENTS)} logged events; fixture quotes verified")

In [ ]:
def show_transcript(messages):
    for m in messages:
        who = "USER" if m["role"] == "user" else " bot"
        print(f"[{m['turn']:2d}] {who}  {m['content']}")

show_transcript(MESSAGES)
print("\nLogged events:")
for e in EVENTS:
    print(f"  {e['id']:<20} turn {e['turn']:2d}  {e['type']:<16} {e['note']}")
tokens = sum(m["usage"]["total_tokens"] for m in MESSAGES)
latency = sum(m["usage"]["latency_ms"] for m in MESSAGES) / 1000
print(f"\nTotals: {tokens} tokens, {latency:.1f}s model latency, "
      f"{RUN['turns_used']} turns, stop={RUN['stop_reason']}")

## The honest generator

In production the report writer is an LLM prompt. Here it is deterministic code, so you can see exactly where honesty comes from — and, in experiments 3–5, exactly where the lies get in. Two rules do all the work:

1. **Quotes are copied verbatim** from the run record, never paraphrased.
2. **The outcome is read from the run record**, never inferred from how the final message *felt*.

The slots are fixed: goal (the user's own words) / outcome / annotated moments / safety / next step / cost + trace pointer.

In [ ]:
def write_report(messages, events, run):
    """Deterministic stand-in for the LLM report writer. Honest by construction."""
    first_user = next(m for m in messages if m["role"] == "user")
    goal_quote = first_user["content"].split(". ")[0]           # verbatim by construction
    last_assistant = [m for m in messages if m["role"] == "assistant"][-1]
    sentences = last_assistant["content"].split(". ")
    suggestion = sentences[1] if len(sentences) > 1 else sentences[0]
    outcome_notes = {
        "task_complete": "The user confirmed completion.",
        "turn_cap": "INCOMPLETE - the turn cap fired mid-task.",
    }
    return {
        "session_id": run["session_id"],
        "goal": {"turn": first_user["turn"], "quote": goal_quote},
        "outcome": {"stop_reason": run["stop_reason"],          # from the record, not the vibe
                    "turns_used": run["turns_used"],
                    "note": outcome_notes.get(run["stop_reason"], "Stopped.")},
        "moments": [{"id": e["id"], "turn": e["turn"], "quote": e["quote"], "note": e["note"]}
                    for e in events],
        "safety": [{"id": e["id"], "turn": e["turn"], "quote": e["quote"], "note": e["note"]}
                   for e in events if e["type"].startswith("safety")],
        "next_step": {"quote": suggestion,
                      "note": "the assistant's closing suggestion, quoted from its final turn"},
        "cost": {"tokens": sum(m["usage"]["total_tokens"] for m in messages),
                 "latency_s": sum(m["usage"]["latency_ms"] for m in messages) / 1000,
                 "turns": run["turns_used"]},
        "trace": f"traces/{run['session_id']}.jsonl",
    }

def render_md(report):
    g, o = report["goal"], report["outcome"]
    lines = [f"# Session debrief - {report['session_id']}", "",
             f"**Goal (your words, turn {g['turn']}):** \"{g['quote']}\"",
             f"**Outcome:** `{o['stop_reason']}` - {o['turns_used']} turns. {o['note']}", "",
             "**What happened:**"]
    lines += [f"- turn {m['turn']}: \"{m['quote']}\" - {m['note']}" for m in report["moments"]]
    lines.append("")
    if report["safety"]:
        lines.append("**Safety:**")
        lines += [f"- turn {s['turn']}: \"{s['quote']}\" - {s['note']}" for s in report["safety"]]
    else:
        lines.append("**Safety:** none logged.")                # absence is stated, never silent
    n = report["next_step"]
    lines += ["", f"**Next step:** {n['note']} (\"{n['quote']}\")"]
    c = report["cost"]
    lines += [f"**Cost:** {c['tokens']} tokens, {c['latency_s']:.1f}s model latency, {c['turns']} turns.",
              f"**Raw trace:** `{report['trace']}`"]
    return "\n".join(lines)

print(render_md(write_report(MESSAGES, EVENTS, RUN)))

## Experiment 1 — the thirty-second review test

The report's acceptance test is behavioral: a reader who has *not* read the transcript answers six questions from the report alone, timed:

1. What was the user trying to do?
2. Did it finish — and if it stopped, why?
3. What actually happened (the moments that mattered)?
4. Did anything safety-relevant happen?
5. What should happen next?
6. What did it cost, and where is the raw evidence?

**Predict first:** which of the six will be answerable from the report alone? Then run the attempt cell, read ONLY the printed report (resist scrolling up), and fill in `MY_ANSWERS`.

In [ ]:
report = write_report(MESSAGES, EVENTS, RUN)
print(render_md(report))

# YOUR TURN: answer from the report ALONE, and time yourself. Thirty seconds is the bar.
MY_ANSWERS = {
    "goal": "",       # what was the user trying to do?
    "outcome": "",    # did it finish? if it stopped, why?
    "moments": "",    # what actually happened - the moments that mattered?
    "safety": "",     # did anything safety-relevant happen?
    "next_step": "",  # what should happen next?
    "proof": "",      # what did it cost? where is the raw evidence?
}

In [ ]:
# SOLUTION
REVIEWER_QUESTIONS = [
    ("goal", "What was the user trying to do?"),
    ("outcome", "Did it finish - and if it stopped, why?"),
    ("moments", "What actually happened (the moments that mattered)?"),
    ("safety", "Did anything safety-relevant happen?"),
    ("next_step", "What should happen next?"),
    ("proof", "What did it cost, and where is the raw evidence?"),
]

def section_check(report):
    """Deterministic proxy for 'answerable from the report': each slot present and non-empty.
    An explicit 'none logged' counts as an answer - silence would not."""
    c = report["cost"]
    return {
        "goal": report["goal"]["quote"] or None,
        "outcome": report["outcome"]["stop_reason"] or None,
        "moments": (f"{len(report['moments'])} annotated moments"
                    if len(report["moments"]) >= 2 else None),
        "safety": f"{len(report['safety'])} event(s)" if report["safety"] else "none logged",
        "next_step": report["next_step"]["quote"] or None,
        "proof": (f"{c['tokens']} tokens; trace {report['trace']}"
                  if c["tokens"] > 0 and report["trace"] else None),
    }

answers = section_check(report)
answered = 0
for key, question in REVIEWER_QUESTIONS:
    ok = answers[key] is not None
    answered += ok
    print(f"{'ok ' if ok else 'MISS'}  {question}\n      -> {answers[key]}")
print(f"\n{answered}/6 questions answerable from the report alone.")
# If your timed review missed one of these, that is a report defect, not a reader defect.

## Experiment 2 — the chronological recap

The alternative everyone actually ships first: a faithful, complete, turn-by-turn summary. It contains nearly everything the report contains.

**Predict first:** how many of the six questions will it answer *structurally* — labeled, findable in seconds by a reader who skims? And where in the pile does the safety moment land?

In [ ]:
# The recap generator everyone ships first: faithful, chronological, unread.
def write_recap(messages):
    lines = ["# Session recap", ""]
    for m in messages:
        text = m["content"] if len(m["content"]) <= 80 else m["content"][:77] + "..."
        lines.append(f"- turn {m['turn']} ({m['role']}): {text}")
    return "\n".join(lines)

recap = write_recap(MESSAGES)
print(recap)
print(f"\n({len(recap)} characters, every turn weighted equally)")

In [ ]:
# SOLUTION: run the six-question check against the recap.
# Proxy: a question is answerable at a glance iff a labeled slot exists for it.
LABELS = {"goal": "Goal", "outcome": "Outcome", "moments": "What happened",
          "safety": "Safety", "next_step": "Next step", "proof": "Cost"}
report_md = render_md(report)
print(f"{'question':<58}{'recap':<8}{'report'}")
for key, question in REVIEWER_QUESTIONS:
    in_recap = "yes" if LABELS[key] in recap else "no"
    in_report = "yes" if LABELS[key] in report_md else "no"
    print(f"{question:<58}{in_recap:<8}{in_report}")

# The safety moment IS in the recap - technically. Where?
pos = next(i for i, line in enumerate(recap.splitlines()) if "wire" in line) + 1
total = len(recap.splitlines())
print(f"\nThe safety moment sits at line {pos} of {total}, weighted like the small talk.")
print("No stop reason, no cost, no trace pointer: faithful, complete, and unusable in 30 seconds.")

## Experiment 3 — the citation validator

The production report writer is an LLM, and its characteristic lie is the *tidy paraphrase presented as a quote* — fluent, same meaning, not what was said. The `write_report_sloppy` variant below does exactly that to two quotes.

**Predict first:** (a) will your validator catch both paraphrases? (b) A weaker validator that only checks "does the cited turn number exist" — how many of the sloppy report's lies survive it?

In [ ]:
def validate_citations(report, messages, run):
    """Every claim must resolve. Return a list of violations (empty = clean)."""
    by_turn = {m["turn"]: m for m in messages}
    violations = []
    # YOUR ATTEMPT - three rules:
    #  1. the goal quote is a verbatim substring of its cited turn
    #  2. every moment quote AND every safety quote is a verbatim substring of ITS cited turn
    #  3. the stated stop_reason matches the run record
    return violations

# The honest report should pass:
problems = validate_citations(report, MESSAGES, RUN)
print("honest report:", problems or "clean")
# CAREFUL: an empty validator prints "clean" for everything.
# That is S02's fixture invariant staring at you - prove the checker can fail.

In [ ]:
# SOLUTION
def validate_citations(report, messages, run):
    by_turn = {m["turn"]: m for m in messages}
    violations = []
    g = report["goal"]
    if g["turn"] not in by_turn:
        violations.append(f"goal cites turn {g['turn']}, which does not exist")
    elif g["quote"] not in by_turn[g["turn"]]["content"]:
        violations.append(f"goal quote not found in turn {g['turn']}")
    for slot in ("moments", "safety"):
        for m in report[slot]:
            if m["turn"] not in by_turn:
                violations.append(f"{slot} claim cites turn {m['turn']}, which does not exist")
            elif m["quote"] not in by_turn[m["turn"]]["content"]:
                violations.append(f"{slot} quote not found in turn {m['turn']}: \"{m['quote']}\"")
    if report["outcome"]["stop_reason"] != run["stop_reason"]:
        violations.append(f"outcome claims {report['outcome']['stop_reason']!r}, "
                          f"run record says {run['stop_reason']!r}")
    return violations

print("honest report:", validate_citations(report, MESSAGES, RUN) or "clean")

SLOPPY_QUOTES = {  # the LLM-flavored failure: fluent tidying, presented as verbatim
    "a wire is probably not in there": "there is probably no wire in that spot",
    "I did not need the torque note this time": "the torque note was not needed",
}
def write_report_sloppy(messages, events, run):
    tidied = [dict(e, quote=SLOPPY_QUOTES.get(e["quote"], e["quote"])) for e in events]
    return write_report(messages, tidied, run)

sloppy = write_report_sloppy(MESSAGES, EVENTS, RUN)
print("\nsloppy report:")
for v in validate_citations(sloppy, MESSAGES, RUN):
    print("  VIOLATION:", v)

def validate_citations_weak(report, messages, run):
    """Bounds-only: cited turn numbers must exist; quotes unchecked."""
    by_turn = {m["turn"]: m for m in messages}
    return [f"turn {m['turn']} does not exist"
            for m in report["moments"] if m["turn"] not in by_turn]

print("\nsloppy report, bounds-only validator:",
      validate_citations_weak(sloppy, MESSAGES, RUN) or "clean (every lie survived)")
print("\nA validator that never reads the quotes certifies nothing about them.")

## Experiment 4 — the omission lie

The next variant, `write_report_reassuring`, keeps every remaining sentence accurate — every quote verbatim, every reference correct — and simply never mentions the safety events.

**Predict first:** (a) what does the citation validator say about it? (b) What would a validator have to *read* to catch a lie that leaves no false sentence on the page?

In [ ]:
def write_report_reassuring(messages, events, run):
    """Every sentence accurate. The safety events simply never happened."""
    upbeat = [e for e in events if not e["type"].startswith("safety")]
    return write_report(messages, upbeat, run)

reassuring = write_report_reassuring(MESSAGES, EVENTS, RUN)
print("reassuring report, citation validator:",
      validate_citations(reassuring, MESSAGES, RUN) or "clean (nothing on the page is false)")

def validate_coverage(report, events, run):
    """Every logged event must surface. Return violations (empty = clean)."""
    violations = []
    # YOUR ATTEMPT - two rules, both derived from the RUN RECORD, not the report:
    #  1. every logged event's ID appears in report["moments"]
    #  2. every event of type safety* ALSO appears, by ID, in report["safety"]
    return violations

print("reassuring report, coverage validator:",
      validate_coverage(reassuring, EVENTS, RUN) or "clean (did you implement the checks?)")

In [ ]:
# SOLUTION
def validate_coverage(report, events, run):
    moment_ids = {m["id"] for m in report["moments"]}
    safety_ids = {s["id"] for s in report["safety"]}
    violations = []
    for e in events:
        if e["id"] not in moment_ids:
            violations.append(f"logged {e['type']} event '{e['id']}' (turn {e['turn']}) never surfaced")
        if e["type"].startswith("safety") and e["id"] not in safety_ids:
            violations.append(f"SAFETY event '{e['id']}' (turn {e['turn']}) absent from the safety slot")
    return violations

print("honest report, coverage:", validate_coverage(report, EVENTS, RUN) or "clean")
print("\nreassuring report, coverage:")
for v in validate_coverage(reassuring, EVENTS, RUN):
    print("  VIOLATION:", v)

print("\nThe two validators are complementary - each passes reports the other fails:")
print(f"{'variant':<14}{'citation':<12}{'coverage'}")
for name, r in [("honest", report), ("sloppy", sloppy), ("reassuring", reassuring)]:
    c = "clean" if not validate_citations(r, MESSAGES, RUN) else "DIRTY"
    v = "clean" if not validate_coverage(r, EVENTS, RUN) else "DIRTY"
    print(f"{name:<14}{c:<12}{v}")
print("\nHonesty is the conjunction: every claim resolves AND every logged event id surfaces.")
print("Coverage now compares event IDENTITIES, not turn numbers: the wrong event at the")
print("right turn fails on id mismatch, and two events sharing one turn can no longer")
print("mask an omission. Still uncaught: an unsupported INTERPRETATION of an event that")
print("did surface — no deterministic validator sees it; that stays with the judged tier.")

## Experiment 5 — the failed run

Same product, worse day: a heavy mirror on old plaster, and the run hits the turn cap mid-task. The honest generator must say so — the outcome slot is read from the run record, and `turn_cap` does not round up to `task_complete` no matter how close the user felt.

**Predict first:** (a) what does the honest report's outcome slot say? (b) The `write_report_rounded_up` variant declares the run complete — which validator catches it? (c) Its note says the user is "well on their way" — which validator catches *that*?

In [ ]:
FAILED_MESSAGES = [
    {"turn": 1, "role": "user",
     "content": "Heavy mirror, maybe 12 kg, going on an old plaster wall. What hardware?",
     "usage": {"total_tokens": 19, "latency_ms": 0}},
    {"turn": 2, "role": "assistant",
     "content": "For old plaster, toggle bolts, and drill slowly - plaster crumbles. "
                "Mark the line, two bolts minimum.",
     "usage": {"total_tokens": 201, "latency_ms": 890}},
    {"turn": 3, "role": "user",
     "content": "Drilled the first hole and the plaster crumbled around it. The hole is ragged.",
     "usage": {"total_tokens": 19, "latency_ms": 0}},
    {"turn": 4, "role": "assistant",
     "content": "Clean the ragged edge and use a larger washer to spread the load. "
                "Keep the drill speed low.",
     "usage": {"total_tokens": 188, "latency_ms": 810}},
    {"turn": 5, "role": "user",
     "content": "The toggle spun and will not bite. Third hole now.",
     "usage": {"total_tokens": 15, "latency_ms": 0}},
    {"turn": 6, "role": "assistant",
     "content": "If the toggle spins, hold it with needle-nose pliers while you thread "
                "the bolt. Do not keep drilling new holes.",
     "usage": {"total_tokens": 232, "latency_ms": 990}},
    {"turn": 7, "role": "user",
     "content": "Tried that, and the bolt threads stripped. This is not working.",
     "usage": {"total_tokens": 15, "latency_ms": 0}},
    {"turn": 8, "role": "assistant",
     "content": "Stop for now. The right move is a different anchor class entirely: "
                "a rail anchor spreads the load across undamaged plaster.",
     "usage": {"total_tokens": 247, "latency_ms": 1050}},
]

FAILED_EVENTS = [
    {"id": "third-hole", "turn": 5, "type": "observation",
     "quote": "Third hole now",
     "note": "the toggle is not biting; the approach is not converging"},
    {"id": "threads-stripped", "turn": 7, "type": "setback",
     "quote": "the bolt threads stripped",
     "note": "hardware failure; user is stuck"},
]

FAILED_RUN = {"session_id": "mirror-001", "stop_reason": "turn_cap",
              "turns_used": 8, "max_turns": 8, "model": "mock-diy-v1"}

_by_turn = {m["turn"]: m for m in FAILED_MESSAGES}
for _e in FAILED_EVENTS:
    assert _e["quote"] in _by_turn[_e["turn"]]["content"], f"fixture broken at turn {_e['turn']}"

failed_report = write_report(FAILED_MESSAGES, FAILED_EVENTS, FAILED_RUN)
print(render_md(failed_report))

In [ ]:
# SOLUTION: even a failed run gets an honest report - and both validators hold on it.
print("failed-run report, citation:", validate_citations(failed_report, FAILED_MESSAGES, FAILED_RUN) or "clean")
print("failed-run report, coverage:", validate_coverage(failed_report, FAILED_EVENTS, FAILED_RUN) or "clean")

def write_report_rounded_up(messages, events, run):
    """The kindest lie: report the capped run as a success."""
    r = write_report(messages, events, run)
    r["outcome"]["stop_reason"] = "task_complete"
    r["outcome"]["note"] = "The user is well on their way."
    return r

rounded = write_report_rounded_up(FAILED_MESSAGES, FAILED_EVENTS, FAILED_RUN)
print("\nrounded-up report, citation validator:")
for v in validate_citations(rounded, FAILED_MESSAGES, FAILED_RUN):
    print("  VIOLATION:", v)
print("rounded-up report, coverage validator:",
      validate_coverage(rounded, FAILED_EVENTS, FAILED_RUN) or "clean (it kept the moments - the lie is the outcome)")

print("\nBut 'well on their way' is an unsupported quality judgment, not a quote or an event -")
print("no deterministic validator sees it. That is the judged tier, and it waits for S12.")

## What transfers

- The production report writer is a prompt, not Python — but its lies are exactly these: tidy paraphrases presented as quotes, dropped events, outcomes rounded up. The toy made them deterministic so you could watch validators catch them.
- Citation + coverage validators are the **deterministic tier around the report writer** — S02's two-tier rule applied to reports. They are cheap; run them on every report, not on a sample.
- Fixed slots, surprises first, an explicit "none logged", and a trace pointer: the difference between a report and a recap is measurable — six questions, thirty seconds.
- What the toy cannot check: whether the next step was *good*, whether "well on their way" was fair. Unsupported quality judgments need the judged tier, and the judged tier needs calibration (S12) before it can be trusted with anything.
- The thirty-second review test is the acceptance bar: run it on your own report after a real session, depleted, timed — then open the raw log and check the report didn't lie.

Now do the real build in your own project. You type it.